# Tutorial: Colab GPU Training for Dog Pose

[Open in Colab](https://colab.research.google.com/github/youhs4554/dog-pose/blob/main/notebooks/colab_dog_pose_training.ipynb)

Audience:
- Developers who want to run the Ultralytics dog-pose project on a Colab GPU instead of a local Mac.

Prerequisites:
- A Colab runtime with GPU enabled.
- Google Drive space for checkpoints.
- A GitHub repo that contains this project.

Learning goals:
- By the end, you can launch a long GPU training run, resume after Colab disconnects, and inspect the saved checkpoint.


## Outline

1. Mount Google Drive and sync the repo into Colab.
2. Install the project with `uv` without relying on the local Mac lockfile.
3. Launch or resume a long dog-pose training run on CUDA.
4. Inspect the saved checkpoint and run a quick inference smoke test.

Resume rule:
- Keep `RUN_NAME` fixed and rerun the training cell.
- Keep `TARGET_TOTAL_EPOCHS` at the total epoch target you want.
- If a 100-epoch run finishes and you want 200 total epochs, change `TARGET_TOTAL_EPOCHS = 200` and run the training cell again.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/youhs4554/dog-pose.git"
REPO_BRANCH = "main"
WORKDIR = Path("/content/dog-pose")
DRIVE_ROOT = Path("/content/drive/MyDrive/dog-pose-colab")
PROJECT_DIR = DRIVE_ROOT / "runs" / "pose"
DATASETS_DIR = Path("/content/datasets")
RUN_NAME = "dog-pose-colab-longrun"
TARGET_TOTAL_EPOCHS = 100
BATCH = "16"
IMG_SIZE = 640
SAVE_PERIOD = 5
WORKERS = 2


def run(*cmd: str) -> None:
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)


## Step 1 - Mount Drive and sync the repository

This notebook expects the project to live in GitHub. If the repo is private, replace `REPO_URL` with an authenticated clone URL before running this cell.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

if WORKDIR.exists():
    run("git", "-C", str(WORKDIR), "fetch", "--all", "--prune")
    run("git", "-C", str(WORKDIR), "checkout", REPO_BRANCH)
    run("git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", REPO_BRANCH)
else:
    run("git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(WORKDIR))

os.chdir(WORKDIR)
print("working directory:", Path.cwd())
print("drive root:", DRIVE_ROOT)


## Step 2 - Install dependencies with `uv` and confirm the GPU runtime

Use `uv` for package installation, but avoid `uv sync` here because the local lockfile was generated on macOS and Colab should resolve against its Linux GPU environment.


In [ ]:
run(sys.executable, "-m", "pip", "install", "-q", "uv")
run("uv", "pip", "install", "--system", "-e", ".")

import torch

assert torch.cuda.is_available(), "Enable a GPU runtime in Colab before training."
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0))


## Step 3 - Launch or resume the long training run

This cell calls `scripts/train_dog_pose_colab.py`.

Why this survives long runs better:
- checkpoints are written into Google Drive
- `last.pt` is saved every `SAVE_PERIOD` epochs
- rerunning this exact cell resumes from `last.pt` when it exists
- `patience` is held high enough to avoid stopping before `TARGET_TOTAL_EPOCHS`


In [ ]:
train_cmd = [
    sys.executable,
    "scripts/train_dog_pose_colab.py",
    "--epochs", str(TARGET_TOTAL_EPOCHS),
    "--batch", str(BATCH),
    "--imgsz", str(IMG_SIZE),
    "--workers", str(WORKERS),
    "--device", "0",
    "--project", str(PROJECT_DIR),
    "--name", RUN_NAME,
    "--save-period", str(SAVE_PERIOD),
    "--datasets-dir", str(DATASETS_DIR),
    "--cache", "disk",
    "--resume",
    "--exist-ok",
]
# print(" ".join(train_cmd))
!{" ".join(train_cmd)}


## Step 4 - Inspect the saved checkpoint

The training script writes a `latest_run.json` file next to the Colab run directory so the next cells can find the newest checkpoint deterministically.


In [ ]:
summary_path = PROJECT_DIR / "latest_run.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary


## Step 5 - Run a quick inference smoke test

This verifies that the saved dog-pose checkpoint can be loaded and rendered with the same skeleton overlay logic used by the Streamlit app.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

from dog_pose_mvp.visualization import (
    draw_pose_result,
    load_image,
    load_model,
    predict_image,
    result_to_records,
)

val_dir = DATASETS_DIR / "dog-pose" / "images" / "val"
sample_image = sorted(val_dir.glob("*.jpg"))[0]
model = load_model(summary["best"])
image = Image.open(sample_image)
image_bgr = load_image(image)
result = predict_image(model, image_bgr, conf=0.25, imgsz=IMG_SIZE)[0]
rendered = draw_pose_result(image_bgr, result, keypoint_conf=0.10)

print(f"sample_image={sample_image}")
print(f"detections={len(result.boxes) if result.boxes is not None else 0}")
print(f"visible_joints={len(result_to_records(result, 0.10))}")

plt.figure(figsize=(10, 8))
plt.imshow(rendered[:, :, ::-1])
plt.axis("off")
plt.show()


## Pitfalls and extensions

Common mistake:
- Changing `RUN_NAME` starts a brand-new run instead of resuming.

How to continue past the original target:
- Increase `TARGET_TOTAL_EPOCHS` and rerun only the training cell.

Optional extension:
- Try `BATCH = "0.70"` to let Ultralytics target roughly 70% of the CUDA memory instead of a fixed integer batch size.


In [ ]:
# Optional exercise scaffold
# 1. Copy the training cell.
# 2. Set RUN_NAME = "dog-pose-colab-ablation".
# 3. Change IMG_SIZE or BATCH.
# 4. Compare the new run directory inside PROJECT_DIR.
